# 01 — Cross-Sectional Analysis (Q1)

**Research Question:** Does baseline dopaminergic brain structure predict baseline PQ-BC severity?

Pipeline: Nested 5-fold CV with per-fold ICV ratio → ComBat → StandardScaler → SVR (linear kernel).

Two datasets:
- **Structural-only** (12 features): 6 bilateral subcortical volumes
- **Combined** (14 features): structural + 2 SCS MD connectivity features (dMRI QC subset)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

from src.core.config import initialize_notebook
env = initialize_notebook(run_name='regression')

## 1. Load data

In [ ]:
from src.core.regression.pipeline import load_full_dataset

full_df = load_full_dataset(env)
print(f'Full dataset: {full_df.shape}')

## 2. Define two datasets

In [ ]:
from src.core.features import get_roi_columns_from_config

# --- Structural-only: override connectivity to empty ---
env_struct = deepcopy(env)
env_struct.configs.data['roi_features']['dopamine_core']['connectivity'] = []
struct_cols = get_roi_columns_from_config(env_struct.configs.data, ['dopamine_core'])
print(f'Structural-only features: {len(struct_cols)}')
print(f'Structural-only N: {len(full_df)}')

# --- Combined: filter for dMRI QC ---
dmri_col = 'mr_y_qc__incl__dmri_indicator'
combined_df = full_df[full_df[dmri_col].astype(str) == '1'].copy().reset_index(drop=True)
combined_cols = get_roi_columns_from_config(env.configs.data, ['dopamine_core'])
print(f'\nCombined features: {len(combined_cols)}')
print(f'Combined N (dMRI QC pass): {len(combined_df)}')

## 3. Structural-only SVR (main result)

In [ ]:
from src.core.regression.pipeline import run_target_with_nested_cv

# Use structural-only env (no connectivity features), raw L/R columns
target_config = env_struct.configs.regression['targets'][0]  # pps_severity
print(f'Target: {target_config["name"]} ({target_config["column"]})')

results_struct = run_target_with_nested_cv(
    env_struct, full_df, target_config, model_name='svr', verbose=True
)

r_struct = results_struct['svr']['overall']['pearson_r']
n_struct = results_struct['svr']['n_samples']
print(f'\nStructural-only SVR: r = {r_struct:.4f}, n = {n_struct}')

In [ ]:
# Actual vs Predicted scatter — structural-only SVR
from scipy.stats import pearsonr
from scipy import stats as sp_stats

if 'y_true' in results_struct['svr']:
    y_true_s = results_struct['svr']['y_true']
    y_pred_s = results_struct['svr']['y_pred']
else:
    # Fallback: reconstruct from saved fold data
    import pickle
    seed = env_struct.configs.run['seed']
    reg_dir = (
        env_struct.repo_root / 'outputs' / env_struct.configs.run['run_name']
        / env_struct.configs.run['run_id'] / f'seed_{seed}' / 'regression'
        / target_config['name'] / 'svr'
    )
    with open(reg_dir / 'results.pkl', 'rb') as f:
        saved = pickle.load(f)
    folds = saved['svr_folds']
    y_true_s = np.concatenate([f['y_test'] for f in folds])
    y_pred_s = np.concatenate([f['y_pred'] for f in folds])

rv, pv = pearsonr(y_true_s, y_pred_s)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true_s, y_pred_s, alpha=0.4, s=30, color='steelblue', edgecolors='navy', lw=0.5)
z = np.polyfit(y_true_s, y_pred_s, 1); p_line = np.poly1d(z)
xs = np.sort(y_true_s)
ax.plot(xs, p_line(xs), 'r-', lw=2.5, zorder=9)
# 95% CI band
res = y_pred_s - p_line(y_true_s)
se = np.sqrt(np.sum(res**2) / (len(y_true_s) - 2))
tcrit = sp_stats.t.ppf(0.975, len(y_true_s) - 2)
ci = tcrit * se * np.sqrt(1/len(y_true_s) + (xs - y_true_s.mean())**2 / np.sum((y_true_s - y_true_s.mean())**2))
ax.fill_between(xs, p_line(xs) - ci, p_line(xs) + ci, color='red', alpha=0.12)

pstr = 'p < 0.001' if pv < 0.001 else f'p = {pv:.4f}'
ax.text(0.05, 0.97, f'r = {rv:.3f} ({pstr})\nn = {len(y_true_s)}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9), family='monospace')
ax.set_xlabel('Observed (residualized)', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted', fontsize=13, fontweight='bold')
ax.set_title('Structural-only SVR — PPS Severity (Q1)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 4. Combined SVR (structural + DTI)

In [ ]:
# Use full env (with SCS connectivity features), on dMRI QC subset
env_combined = deepcopy(env)

results_combined = run_target_with_nested_cv(
    env_combined, combined_df, target_config, model_name='svr', verbose=True
)

r_combined = results_combined['svr']['overall']['pearson_r']
n_combined = results_combined['svr']['n_samples']
print(f'\nCombined SVR: r = {r_combined:.4f}, n = {n_combined}')
print(f'Structural-only SVR: r = {r_struct:.4f}, n = {n_struct}')

## 5. Sex-stratified SVR

In [ ]:
from src.core.regression.robustness import sex_stratified_svr

print('--- Structural-only ---')
sex_results_struct = sex_stratified_svr(
    env_struct, full_df, target_config, model_name='svr'
)

print('\n--- Combined ---')
sex_results_combined = sex_stratified_svr(
    env_combined, combined_df, target_config, model_name='svr'
)

## 6. Lateralization comparison

In [ ]:
# Lateralization comparison: run same pipeline with different feature_transform settings
# raw (Original L/R), asymmetry (AI only), total (Total volume only), ai_total (AI + Total)

lat_transforms = {
    'Original L/R': 'raw',
    'Asymmetry only (AI)': 'asymmetry',
    'Total volume only': 'total',
    'AI + Total': 'ai_total',
}

lat_results = {}
for label, transform in lat_transforms.items():
    env_lat = deepcopy(env_struct)
    env_lat.configs.regression['feature_transform'] = transform
    res = run_target_with_nested_cv(
        env_lat, full_df, target_config, model_name='svr', verbose=False
    )
    r = res['svr']['overall']['pearson_r']
    p = res['svr']['overall'].get('pearson_p_parametric', np.nan)
    lat_results[label] = {'r': r, 'p': p, 'y_true': res['svr']['y_true'], 'y_pred': res['svr']['y_pred']}
    print(f'  {label:<25} r = {r:+.4f}, p = {p:.4f}')

print(f"\n{'Transform':<30} {'r':>8} {'p':>10}")
print('-' * 50)
for set_name, res in lat_results.items():
    print(f"{set_name:<30} {res['r']:>+8.4f} {res['p']:>10.4f}")

In [ ]:
# Lateralization comparison bar chart
lat_names = list(lat_results.keys())
lat_rs = [lat_results[k]['r'] for k in lat_names]
lat_ps = [lat_results[k]['p'] for k in lat_names]

colors = ['#2ca02c' if p < 0.05 else '#d9d9d9' for p in lat_ps]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(range(len(lat_names)), lat_rs, color=colors, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_yticks(range(len(lat_names)))
ax.set_yticklabels(lat_names, fontsize=11)
ax.set_xlabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Lateralization Comparison — PPS Severity', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=1)
for i, (r, p) in enumerate(zip(lat_rs, lat_ps)):
    sig = '*' if p < 0.05 else ''
    ax.text(max(r, 0) + 0.003, i, f'{r:.3f} (p={p:.3f}){sig}', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 7. Univariate AI correlations + Volume vs Asymmetry

In [ ]:
from src.core.regression.univariate import (
    extract_bilateral_pairs, prepare_harmonized_data, compute_asymmetry_features,
    univariate_correlations, volume_vs_asymmetry_tests,
)

# Harmonize for univariate analysis
bilateral_pairs, unpaired = extract_bilateral_pairs(env_struct.configs.data, ['dopamine_core'])

X_harm, y_harm, df_harm, valid_cols = prepare_harmonized_data(
    full_df, struct_cols,
    harmonize_config=env_struct.configs.harmonize,
    regression_config=env_struct.configs.regression,
    target_col=target_config['column'],
    target_name=target_config['name'],
    data_config=env_struct.configs.data,
)

valid_pairs = [(n, l, r) for n, l, r in bilateral_pairs
               if l in valid_cols and r in valid_cols]

# AI correlations with FDR + Bonferroni corrections
asym = compute_asymmetry_features(X_harm, valid_cols, valid_pairs)
ai_keys = sorted(k for k in asym if k.endswith('_AI'))

X_ai = np.column_stack([asym[k] for k in ai_keys])
corr_df = univariate_correlations(X_ai, y_harm, ai_keys, corrections=('bonferroni', 'fdr_bh'))
print('Univariate AI correlations (FDR + Bonferroni corrected):')
print(corr_df[['feature', 'r', 'p_raw', 'p_bonferroni', 'sig_bonferroni',
               'p_fdr_bh', 'sig_fdr_bh']].to_string(index=False))

In [ ]:
# Univariate AI correlation forest plot
from src.core.regression.robustness import bootstrap_feature_ci

boot_df = bootstrap_feature_ci(X_harm, y_harm, valid_pairs, valid_cols, n_boot=10000, seed=42)

fig, ax = plt.subplots(figsize=(8, max(4, len(boot_df) * 0.6)))
ypos = np.arange(len(boot_df))
colors = ['#d62728' if p < 0.05 else '#1f77b4' if p < 0.1 else '#d9d9d9'
          for p in boot_df['p_boot_fdr']]
xerr_lo = boot_df['r_obs'].values - boot_df['ci_lo'].values
xerr_hi = boot_df['ci_hi'].values - boot_df['r_obs'].values

ax.barh(ypos, boot_df['r_obs'].values, xerr=[xerr_lo, xerr_hi],
        color=colors, edgecolor='black', lw=0.4, alpha=0.8,
        error_kw={'lw': 1.2, 'ecolor': 'black', 'capsize': 3})
ax.axvline(0, color='black', lw=1.2)
ax.set_yticks(ypos)
ax.set_yticklabels(boot_df['feature'].values, fontsize=10)
ax.set_xlabel('Pearson r (bootstrap 95% CI)', fontsize=12, fontweight='bold')
ax.set_title('Univariate AI Correlations with PPS Severity', fontsize=13, fontweight='bold')
for i, row in boot_df.iterrows():
    stars = '***' if row['p_boot_fdr'] < 0.001 else '**' if row['p_boot_fdr'] < 0.01 else '*' if row['p_boot_fdr'] < 0.05 else ''
    ax.text(max(row['ci_hi'], 0) + 0.005, i, stars, va='center', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

In [ ]:
# Steiger tests: AI vs Total volume correlations
steiger_df = volume_vs_asymmetry_tests(X_harm, y_harm, valid_cols, valid_pairs)
print('Volume vs Asymmetry Steiger tests:')
print(steiger_df.to_string(index=False))

## 8. Sex differences (t-tests on AI) + Sex x AI interactions

In [ ]:
from src.core.regression.univariate import sex_differences_anova, sex_interaction_test

if 'sex_mapped' in df_harm.columns:
    sex_labels = df_harm['sex_mapped'].values

    # t-tests on AI features by sex
    sex_diff_df = sex_differences_anova(asym, sex_labels)
    print('Sex differences in AI features:')
    print(sex_diff_df[['feature', 'mean_male', 'mean_female', 't', 'p_raw', 'p_fdr', 'cohens_d']].to_string(index=False))

    # Sex x AI interaction (includes per-sex r and p)
    sex_int_df = sex_interaction_test(asym, y_harm, sex_labels)
    print('\nSex x AI interaction tests:')
    print(sex_int_df[['feature', 'r_male', 'p_male', 'r_female', 'p_female',
                       'beta_interaction', 'p_interaction', 'p_interaction_fdr']].to_string(index=False))

## 9. Permutation test + Bootstrap CI

In [ ]:
from src.core.regression.evaluation import permutation_test, compute_permutation_pvalue, bootstrap_ci

perm_results = permutation_test(
    env_struct, full_df, target_config, model_name='svr',
    n_permutations=env_struct.configs.regression['permutation']['n_permutations'], verbose=True,
)

p_emp = compute_permutation_pvalue(r_struct, perm_results['null_distribution'])
print(f'\nObserved r = {r_struct:.4f}')
print(f'Permutation p = {p_emp:.4f} (n_perm={perm_results["n_permutations"]})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_results['null_distribution'], bins=50, alpha=0.7, edgecolor='black', label='Null')
ax.axvline(r_struct, color='red', linewidth=2, label=f'Observed r = {r_struct:.3f}')
ax.set_xlabel('Pearson r')
ax.set_ylabel('Count')
ax.set_title(f'Permutation Test: p = {p_emp:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bootstrap CI from pooled CV predictions
import pickle

seed = env_struct.configs.run['seed']
reg_dir = (
    env_struct.repo_root / 'outputs' / env_struct.configs.run['run_name'] / env_struct.configs.run['run_id']
    / f'seed_{seed}' / 'regression' / target_config['name'] / 'svr'
)
with open(reg_dir / 'results.pkl', 'rb') as f:
    saved = pickle.load(f)

folds = saved['svr_folds']
y_true = np.concatenate([f['y_test'] for f in folds])
y_pred = np.concatenate([f['y_pred'] for f in folds])

boot_results = bootstrap_ci(y_true, y_pred, n_bootstrap=env_struct.configs.regression['bootstrap']['n_bootstrap'], seed=seed)
for metric, vals in boot_results.items():
    print(f"{metric}: {vals['observed']:.4f} [{vals['lower']:.4f}, {vals['upper']:.4f}]")

## 10. Robustness: one-per-family, feature importance

In [ ]:
from src.core.regression.robustness import one_per_family_permutation

opf = one_per_family_permutation(env_struct, full_df, target_config)

print(f'One-per-family median r: {opf["observed_r"]:.4f}')
print(f'  95% CI: [{opf["ci_lo"]:.4f}, {opf["ci_hi"]:.4f}]')
print(f'  Valid runs: {opf["n_valid"]}')

## 11. Cortical thickness SVR (cortical_dopamine network)

In [ ]:
# Cortical thickness: 20 features from cortical_dopamine network
# ICV correction uses CT / ICV^(1/3)
env_cort = deepcopy(env)
env_cort.configs.regression['roi_networks'] = ['cortical_dopamine']

cort_cols = get_roi_columns_from_config(env_cort.configs.data, ['cortical_dopamine'])
cort_present = [c for c in cort_cols if c in full_df.columns]
print(f'Cortical dopamine features: {len(cort_present)} present / {len(cort_cols)} defined')

if len(cort_present) >= 2:
    results_cort = run_target_with_nested_cv(
        env_cort, full_df, target_config, model_name='svr', verbose=True
    )
    r_cort = results_cort['svr']['overall']['pearson_r']
    n_cort = results_cort['svr']['n_samples']
    print(f'\nCortical thickness SVR: r = {r_cort:.4f}, n = {n_cort}')
    print(f'Structural-only SVR:    r = {r_struct:.4f}, n = {n_struct}')
else:
    print('WARNING: Not enough cortical thickness features present in data')
    r_cort, n_cort = np.nan, 0

## 12. Diffusion-only SVR (SCS MD features)

In [ ]:
# Diffusion-only: 2 SCS MD features (no structural volumes)
env_dti = deepcopy(env)
env_dti.configs.data['roi_features']['dopamine_core']['structural'] = []
dti_cols = get_roi_columns_from_config(env_dti.configs.data, ['dopamine_core'])
print(f'Diffusion-only features: {len(dti_cols)}')

# Require dMRI QC pass
results_dti = run_target_with_nested_cv(
    env_dti, combined_df, target_config, model_name='svr', verbose=True
)
r_dti = results_dti['svr']['overall']['pearson_r']
n_dti = results_dti['svr']['n_samples']
print(f'\nDiffusion-only SVR: r = {r_dti:.4f}, n = {n_dti}')
print(f'Combined SVR:       r = {r_combined:.4f}, n = {n_combined}')

## 13. Per-region SVR

In [ ]:
from src.core.regression.robustness import per_region_svr

print('Per-region SVR (structural-only, each bilateral pair individually):')
region_df = per_region_svr(env_struct, full_df, target_config, model_name='svr')
print(f'\n{region_df[["region", "r", "p_raw", "p_fdr", "n"]].to_string(index=False)}')

In [ ]:
# Per-region SVR bar chart with FDR significance
fig, ax = plt.subplots(figsize=(8, max(4, len(region_df) * 0.6)))
ypos = np.arange(len(region_df))
colors = ['#2ca02c' if p < 0.05 else '#ff7f0e' if p < 0.1 else '#d9d9d9'
          for p in region_df['p_fdr']]
ax.barh(ypos, region_df['r'].values, color=colors, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_yticks(ypos)
ax.set_yticklabels(region_df['region'].values, fontsize=11)
ax.set_xlabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Per-Region SVR — PPS Severity (Q1)', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=1)
for i, row in region_df.iterrows():
    stars = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*' if row['p_fdr'] < 0.05 else ''
    ax.text(row['r'] + 0.003, i, f'r={row["r"]:.3f} {stars}', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 14. Anxiety target (CBCL DSM anxiety)

In [ ]:
# Anxiety target: structural-only SVR with CBCL anxiety
anx_target = {'name': 'cbcl_anxiety', 'column': 'mh_p_cbcl__dsm__anx_sum'}
anx_col = anx_target['column']

if anx_col in full_df.columns and full_df[anx_col].notna().sum() > 50:
    print(f"Anxiety target: {anx_col}")
    print(f"  N with data: {full_df[anx_col].notna().sum()}")
    print(f"  Mean: {full_df[anx_col].dropna().mean():.2f}")

    # Structural-only
    print('\n--- Structural-only -> Anxiety ---')
    results_anx_struct = run_target_with_nested_cv(
        env_struct, full_df, anx_target, model_name='svr', verbose=True
    )
    r_anx_struct = results_anx_struct['svr']['overall']['pearson_r']
    n_anx_struct = results_anx_struct['svr']['n_samples']
    print(f'Structural-only -> Anxiety: r = {r_anx_struct:.4f}, n = {n_anx_struct}')

    # Combined
    print('\n--- Combined -> Anxiety ---')
    results_anx_combined = run_target_with_nested_cv(
        env, combined_df, anx_target, model_name='svr', verbose=True
    )
    r_anx_combined = results_anx_combined['svr']['overall']['pearson_r']
    n_anx_combined = results_anx_combined['svr']['n_samples']
    print(f'Combined -> Anxiety: r = {r_anx_combined:.4f}, n = {n_anx_combined}')
else:
    print(f'WARNING: Anxiety column {anx_col} not available')
    r_anx_struct, n_anx_struct = np.nan, 0
    r_anx_combined, n_anx_combined = np.nan, 0

## 15. Summary table — Q1 all feature sets x both targets

In [ ]:
# Summary table: all feature sets x both targets
summary_rows = [
    {'Feature Set': 'Subcortical (12)', 'Target': 'PPS Severity', 'r': r_struct, 'n': n_struct},
    {'Feature Set': 'Combined (14)', 'Target': 'PPS Severity', 'r': r_combined, 'n': n_combined},
    {'Feature Set': 'Cortical (20)', 'Target': 'PPS Severity', 'r': r_cort, 'n': n_cort},
    {'Feature Set': 'Diffusion-only (2)', 'Target': 'PPS Severity', 'r': r_dti, 'n': n_dti},
    {'Feature Set': 'Subcortical (12)', 'Target': 'CBCL Anxiety', 'r': r_anx_struct, 'n': n_anx_struct},
    {'Feature Set': 'Combined (14)', 'Target': 'CBCL Anxiety', 'r': r_anx_combined, 'n': n_anx_combined},
]

summary_df = pd.DataFrame(summary_rows)
summary_df['r'] = summary_df['r'].map(lambda x: f'{x:+.4f}' if pd.notna(x) else 'N/A')
print('Q1 Cross-Sectional Summary:')
print(summary_df.to_string(index=False))

In [ ]:
# Summary bar chart: all feature sets x both targets
labels = ['Subcortical\n(12)', 'Combined\n(14)', 'Cortical\n(20)', 'Diffusion\n(2)']
pps_rs = [r_struct, r_combined, r_cort, r_dti]
anx_rs = [r_anx_struct, r_anx_combined, np.nan, np.nan]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, pps_rs, width, label='PPS Severity', color='steelblue',
               edgecolor='black', lw=0.5, alpha=0.85)
bars2 = ax.bar(x + width/2, [r if np.isfinite(r) else 0 for r in anx_rs], width,
               label='CBCL Anxiety', color='coral', edgecolor='black', lw=0.5, alpha=0.85)

ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Q1 Cross-Sectional: Feature Set Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, ls='--')

# Annotate r values
for bar, r in zip(bars1, pps_rs):
    if np.isfinite(r):
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.005 * np.sign(r),
                f'{r:.3f}', ha='center', va='bottom' if r >= 0 else 'top', fontsize=9)
for bar, r in zip(bars2, anx_rs):
    if np.isfinite(r):
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.005 * np.sign(r),
                f'{r:.3f}', ha='center', va='bottom' if r >= 0 else 'top', fontsize=9)

plt.tight_layout()
plt.show()